In [1]:
import numpy as np
import torch
from torch_geometric.data import Data

/Users/ryan/Documents/NUS/Y4S1/DSA4266/network-traffic-classification/.venv/lib/python3.11/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


In [2]:
X_train = np.load("../../data/processed/X_train.npy")
pairs_train = np.load("../../data/processed/pairs_train.npy")
y_train = np.load("../../data/processed/y_train.npy")

X_test = np.load("../../data/processed/X_test.npy")
pairs_test = np.load("../../data/processed/pairs_test.npy")
y_test = np.load("../../data/processed/y_test.npy")

# verify if shapes of train, test and pair the same

In [3]:
# mapping each ip addrress to nodes
unique_ips = np.unique(np.concatenate([pairs_train[:, 0],pairs_train[:, 1]]))

node_to_id = {
    ip: i
    for i, ip in enumerate(unique_ips)
}

print("Num of nodes:", len(node_to_id))

Num of nodes: 63


In [4]:
src_ids = np.array([node_to_id[src]for src in pairs_train[:, 0]])
dst_ids = np.array([node_to_id[dst]for dst in pairs_train[:, 1]])

In [5]:
## representation for GCN msg passing
# unique structural connection 
graph_connections = np.unique(np.vstack([src_ids, dst_ids]), axis=1)
reverse_connections = graph_connections[[1, 0], :]

edge_index = np.unique(
    np.concatenate([graph_connections,reverse_connections],axis=1),
    axis = 1)

In [6]:
# flatten preprocessed seq feature to aggregate
X_edge_train = X_train.reshape(X_train.shape[0],-1)

num_nodes = len(node_to_id)
num_features = X_edge_train.shape[1]

# 1 feat vector per node
node_features = np.zeros((num_nodes, num_features), dtype=np.float32)
node_counts = np.zeros(num_nodes,dtype=np.float32)

for i in range(len(X_edge_train)):

    src = src_ids[i]
    dst = dst_ids[i]

    # src and des features
    node_features[src] += X_edge_train[i]
    node_features[dst] += X_edge_train[i]

    node_counts[src] += 1
    node_counts[dst] += 1

# mean feat of all training traffic per ip node
for node in range(num_nodes):
    if node_counts[node] > 0:
        node_features[node] /= node_counts[node]

In [ ]:
x = torch.tensor(node_features, dtype=torch.float32)
print(edge_index.shape)
data = Data(x=x, edge_index=edge_index)

(2, 136)


AttributeError: 'numpy.ndarray' object has no attribute 't'